### RAG Pipeline - Data Ingestion to Vector DB Pipeline


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 5 PDF files to process

Processing: CC & DC – U & S V1.pdf
  ✓ Loaded 6 pages

Processing: Income_Tax_Act_2025_as_amended_by_FA_Act_2026.pdf
  ✓ Loaded 686 pages

Processing: leave.pdf
  ✓ Loaded 11 pages

Processing: poem.pdf
  ✓ Loaded 2 pages

Processing: Website Privacy Policy V2.pdf
  ✓ Loaded 8 pages

Total documents loaded: 713


In [3]:
### Text Splitting get into chunks 

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\n Example chunk:")
        print(f"Content :{split_docs[0].page_content[:200]}")  # Print first 200 characters of the first chunk
        print(f"Metadata :{split_docs[0].metadata}")  # Print metadata of the first chunk
    
    return split_docs

# Split the loaded PDF documents into chunks





    

In [4]:
chunks = split_documents(all_pdf_documents)

Split 713 documents into 2350 chunks

 Example chunk:
Content :Confidential property of Consint.ai. Do not distribute or reproduce without express permission from Consint.ai 
 
Corporate Credit and Debit Cards 
Usage and Settlement
Metadata :{'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf'}


In [5]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf'}, page_content='Confidential property of Consint.ai. Do not distribute or reproduce without express permission from Consint.ai \n \nCorporate Credit and Debit Cards \nUsage and Settlement'),
 Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf'}, page_content='Confidential property of Consint.ai. Do not distribute or reproduce without exp

### embedding and vector store db

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager with a specified model name and load the model.

        Args:
            model_name (str): The name of the sentence transformer model to use for generating embeddings.
            Hugging face model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        Returns:
            np.ndarray: An array of embeddings corresponding to the input texts.
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
        
        print(f"Generating embeddings for {len(texts)} texts...") 
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

    

In [8]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7962.65it/s]


Model 'all-MiniLM-L6-v2' loaded successfully. Embedding dimension: 384


### Class Vector Store

In [9]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name,metadata={"description": "PDF Document for Embeddings","hnsw:space": "cosine"})
            print(f"Vector store initialized at '{self.persist_directory}' with collection '{self.collection_name}'")
            print(f"Collection metadata: {self.collection.metadata} with count of : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        Add documents and their corresponding embeddings to the vector store.

        Args:
            documents (List[Any]): A list of document objects containing metadata.
            embeddings (np.ndarray): An array of embeddings corresponding to the documents.
        """
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        if not self.collection:
            raise ValueError("Vector store not initialized. Call _initialize_store() first.")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chroma 

        ids=[]
        metadatas=[]
        documents_texts=[]
        embeddings_list=[]

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document Content
            documents_texts.append(doc.page_content)

            # Embeddings 
            embeddings_list.append(embedding.tolist())

            # Add to chroma collection
            try:            
                self.collection.add(
                    ids=[doc_id],
                    embeddings=[embedding.tolist()],
                    metadatas=[metadata],
                    documents=[doc.page_content]
                )
                print(f"  ✓ Added document ID: {doc_id} with metadata: {metadata}")
            except Exception as e:
                print(f"  ✗ Error adding document ID: {doc_id} - {e}")
                raise

vectorStore = VectorStore()  

Vector store initialized at '../data/vector_store' with collection 'pdf_documents'
Collection metadata: {'description': 'PDF Document for Embeddings'} with count of : 9706


In [10]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Add the documents and embeddings to the vector store
vectorStore.add_documents(chunks,embeddings)


Generating embeddings for 2350 texts...


Batches: 100%|██████████| 74/74 [00:58<00:00,  1.27it/s]


Generated embeddings with shape: (2350, 384)
Adding 2350 documents to vector store...
  ✓ Added document ID: doc_e1938fcf_0 with metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf', 'doc_index': 0, 'content_length': 168}
  ✓ Added document ID: doc_043f83c9_1 with metadata: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-17T17:38:37+05:30', 'author': 'pc', 'moddate': '2026-04-17T17:38:37+05:30', 'source': '..\\data\\pdf_files\\CC & DC – U & S V1.pdf', 'total_pages': 6, 'page': 1, 'page_label': '2', 'source_file': 'CC & DC – U & S V1.pdf', 'file_type': 'pdf', 'doc_index': 1, 'content_length': 949}
  ✓ Added document ID: doc_4d60a64b_2 with metadata:

### Create Retreiver Pipeline from vector store

In [11]:

class RAGRetriever:
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """ 
        Initialize the retriever

        Args:
            vector_store (VectorStore): An instance of the VectorStore class to retrieve documents from.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class to generate query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents from the vector store based on a query.

        Args:
            query (str): The input query string to search for relevant documents.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum similarity score for retrieved documents.
        """

        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
            
            for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                similarity_score = 1 - distance
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
        except Exception as e:
            print(f"Error occurred while retrieving documents: {e}")

        return retrieved_docs

rag_retriever = RAGRetriever(vectorStore, embedding_manager)



    


In [12]:
rag_retriever.retrieve("When should I take Privileged leave?")

Retrieving documents for query: 'When should I take Privileged leave?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.43it/s]

Generated embeddings with shape: (1, 384)


[{'id': 'doc_4cb16cff_22',
  'content': 'requirement) during current calendar year only. \n \n6 Privilege Leave \nThe company strongly encourages employees to use privilege leave for vacation to ensure a \nhealthy work life balance. \n \n6.1 Employees are expected to plan their privilege leave at least 14 (fourteen) days in advance \nof taking the leave. Privilege leave can be taken only with prior approval of the line  \nmanager. \n6.2 Privilege leave carried forward from the previous year and those earned during a calendar \nyear can be accumulated and utilized any time during the same  calendar year. Any \nunutilized leave more than 30 (thirty) days will lapse at the end of the calendar year and \nonly 30 (thirty) days shall be carried forward to the next calendar year. \n6.3 Encashment of accumulated privilege leaves is available only upon separation  from the \ncompany and the same will be up to a maximum of 30 days.',
  'metadata': {'file_type': 'pdf',
   'author': 'pc',
   'sour

In [13]:
rag_retriever.retrieve("What is Unauthorized Absence")

Retrieving documents for query: 'What is Unauthorized Absence' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.95it/s]


Generated embeddings with shape: (1, 384)


[]

In [14]:

rag_retriever.retrieve("Type of Maternity Leaves")

Retrieving documents for query: 'Type of Maternity Leaves' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.23it/s]

Generated embeddings with shape: (1, 384)


[{'id': 'doc_1b9d486f_27',
  'content': 'delivery or child is handed over to the commissioning/adopting mother, or date of \nmiscarriage/medical termination are eligible for paid maternity leave. For employees with less \nthan 80 (eighty) days service, maternity leave will be treated as unpaid leave. All eligible women \nemployees are entitled to maternity leave, as shown in the table below. The maternity leave \nis inclusive of week offs, and public and national holidays. \n \nType of Maternity \nLeaves \nLeave \nEntitlement (In \nWeeks) \nDocuments required to be \nsubmitted to HR to Avail \nthe Leave \nLeave Commencement \nMaternity Leave in \ncase of women \nemployee up to \ntwo \n \n \n26 \n1. Confirmation of pregnancy \nalong with the date of \ndelivery. \nNot earlier than \neight (8) weeks prior \n(2) surviving \nchildren \n 2. Medical \ncertificate from \ncertified medical \npractitioner. \nto the date\n of delivery. \nMaternity Leave in \ncase of women \nemployee with two \n(2

### Integrate Vectordb Context with LLM output

In [31]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize the Groq model
grok_api_key = os.getenv("GROK_API_KEY")
if not grok_api_key:
    raise ValueError("Groq API key not found. Please set the GROK_API_KEY environment variable.")

llm = ChatGroq(
    api_key=grok_api_key,
    model="openai/gpt-oss-120b",
    temperature=0.7,
    max_tokens=1024,
)

print(f"✓ Groq LLM initialized with model: openai/gpt-oss-120b")

# ===== COMMENTED OUT COHERE CODE =====
# from langchain_cohere import ChatCohere
# import os
# from dotenv import load_dotenv
# 
# # Load environment variables
# load_dotenv()
# 
# # Initialize the Cohere model
# cohere_api_key = os.getenv("COHERE_API_KEY")
# if not cohere_api_key:
#     raise ValueError("Cohere API key not found. Please set the COHERE_API_KEY environment variable.")
# 
# llm = ChatCohere(
#     cohere_api_key=cohere_api_key,
#     model="command-a-03-2025",
#     temperature=0.7,
#     max_tokens=1024,
# )
# 
# print(f"✓ Cohere LLM initialized with model: command-a-03-2025")
# ===== END COMMENTED OUT COHERE CODE =====

### Simple RAG function to generate answer from retrieved documents
def generate_answer(query: str, retriever: RAGRetriever, llm, top_k: int = 5):
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    if not retrieved_docs:
        return "No relevant documents found to answer the query."

    # Combine retrieved documents into a single context, limiting to first 2 docs and 400 chars each
    context_parts = []
    for doc in retrieved_docs[:2]:  # Use only first 2 documents
        content = doc['content'][:400]  # Limit content to 400 characters
        if len(doc['content']) > 400:
            content += "..."
        context_parts.append(f"Document {doc['rank']} (Similarity: {doc['similarity_score']:.2f}):\n{content}")
    
    context = "\n\n".join(context_parts)

    # Create a simple prompt
    prompt = f"Based on this context, answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    # Generate an answer using the Groq model
    try:
        response = llm.invoke(prompt.format(context=context, query=query))
        # Handle different response types
        if hasattr(response, 'content'):
            answer = response.content.strip()
        else:
            answer = str(response).strip()
        
        # Fix escaped newlines in output
        answer = answer.replace('\\n', '\n')
        return answer
    except Exception as e:
        print(f"Error generating answer: {type(e).__name__}: {e}")
        return f"An error occurred while generating the answer."

✓ Groq LLM initialized with model: openai/gpt-oss-120b


In [32]:
answer = generate_answer("What is website security?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is website security?' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.36it/s]

Generated embeddings with shape: (1, 384)


Website security refers to the set of technical, administrative and procedural safeguards put in place to protect a website and the data it handles from unauthorized access, misuse, alteration, disclosure, or loss. It typically includes using a secure hosting infrastructure, continuous monitoring, regular security reviews, and other reasonable measures that meet legal and regulatory requirements, thereby helping prevent data breaches and ensuring that any information shared with third‑party service providers is done only for legitimate purposes.


In [33]:
answer = generate_answer("What is income tax? Explain as if you are explaining to newbie", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is income tax? Explain as if you are explaining to newbie' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.52it/s]

Generated embeddings with shape: (1, 384)


**What is income tax? (A simple, newbie‑friendly explanation)**  

1. **It’s a tax on the money you earn.**  
   - Whenever you receive income—whether it’s a salary, wages, business profits, rent, interest, or any other kind of earnings—the government may require you to pay a portion of that money to the state. That portion is called **income tax**.

2. **How the amount is decided.**  
   - The law (the Income‑Tax Act) says the tax is calculated on your **total income for the whole tax year** (the 12‑month period the government defines).  
   - The tax can be a single rate or a set of rates that increase as your income rises (a “progressive” system).  

3. **What counts as “income tax.”**  
   - The Act also makes clear that **any additional levy that the law calls “income‑tax”**—even if it’s given a different name—is still considered part of your income‑tax liability. In other words, if the government adds a surcharge, a health levy, or any other extra charge on top of the regular tax

In [18]:
answer = generate_answer("Explain me new tax regime", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Explain me new tax regime' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]


Generated embeddings with shape: (1, 384)
### The “New Tax Regime” – an overview (India)

In the Union Budget 2020‑21 the Government introduced an **optional, lower‑rate personal income‑tax structure** that can be chosen by individual taxpayers (including salaried employees, self‑employed professionals, and senior citizens). It runs side‑by‑side with the **old (or “existing”) tax regime**, which retains the higher slab rates but allows a wide range of deductions, exemptions and rebates.

Below is a concise guide to the key features, how it works, and who might benefit.

---

## 1. How the regime is structured

| **Taxable Income (FY 2023‑24)** | **Old Regime**<br>(with deductions/exemptions) | **New Regime**<br>(lower slabs, no most deductions) |
|--------------------------------|-----------------------------------------------|---------------------------------------------------|
| Up to **₹2.5  lakh**           | Nil (basic exemption)                         | Nil (basic exemption)    

In [26]:
answer = generate_answer("Explain Profits in lieu of salary under 100 words summarized.", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Explain Profits in lieu of salary under 100 words summarized.' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.58it/s]

Generated embeddings with shape: (1, 384)


Profits in lieu of salary are taxable cash or non‑cash benefits that a company pays to an employee instead of a regular salary. The amount is treated as salary income for the employee, subject to income‑tax withholding and payroll taxes (such as social security and Medicare in the U.S., or similar levies elsewhere). It includes bonuses, profit‑sharing payouts, commissions, or any profit‑based remuneration that replaces ordinary wages. Because it is considered compensation, the employer must deduct the appropriate taxes and report it on the employee’s wage statements.


In [40]:
answer = generate_answer("Explain  ANTI-AVOIDANCE RULE", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Explain  ANTI-AVOIDANCE RULE' with top_k=5 and score_threshold=0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.75it/s]

Generated embeddings with shape: (1, 384)


**Anti‑avoidance rule – what it is and how it works**

An *anti‑avoidance rule* (often called a **General Anti‑Avoidance Rule** or **GAAR**) is a provision in tax legislation that gives the tax authority the power to disregard or re‑characterise a transaction or arrangement whose main purpose is to obtain a tax benefit that the law was not intended to give.  

---

### 1. Why it exists  

* **Prevent abusive tax planning** – Taxpayers can sometimes structure transactions that are legal in form but are created solely to reduce tax liability.  
* **Protect the tax base** – By stopping “artificial” or “sham” arrangements, the rule helps preserve revenue.  
* **Maintain fairness** – It stops taxpayers from gaining an advantage that ordinary taxpayers, who conduct genuine business, cannot obtain.

---

### 2. Core principle (as reflected in the excerpt)

> “Irrespective of anything contained in this Act, an arrangement entered into by an assessee may be declared to be an impermissible avoid

#### Enhanced Rag Pipeline Features

In [ ]:
def rag_advanced(query: str, retriever: RAGRetriever, llm, top_k: int = 5,min_score: float = 0.2,return_context: bool = False):
    """
    RAG pipeline with extra features:
    - Returns answer,sources,confidence score, and optionally the retrieved context
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    print(f"Retrieved {len(results)} documents for query: '{query}' with min_score={min_score}")

    if not results:
        return {
            "answer": "No relevant documents found to answer the query.",
            "sources": [],
            "confidence": 0,
            "context": [] if return_context else None
        }

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page':doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:200] + ("..." if len(doc['content']) > 200 else "")
    } for doc in results]

    confidence= max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"Based on this context, answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    try:
        response = llm.invoke(prompt.format(context=context, query=query))
        if hasattr(response, 'content'):
            answer = response.content.strip()
        else:
            answer = str(response).strip()
        
        answer = answer.replace('\\n', '\n')
        return {
            "answer": answer,
            "sources": sources,
            "confidence": confidence,
            "context": context if return_context else None
        }
        
    except Exception as e:
        print(f"Error generating answer: {type(e).__name__}: {e}")
        answer = f"An error occurred while generating the answer."


In [54]:
answer = rag_advanced("What is income tax?", rag_retriever, llm,top_k=3,min_score=0.2,return_context=True)
print(f"Answer {answer['answer']}")
print(f"Sources: {answer['sources']}")
print(f"Confidence Score: {answer['confidence']:.2f}")


Retrieving documents for query: 'What is income tax?' with top_k=3 and score_threshold=0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents for query: 'What is income tax?' with min_score=0.2


Answer Income tax is the tax imposed on a person’s or entity’s earnings. In the context of the provision quoted, it is the amount of tax calculated on a “specified sum” (as defined in sections 406 or 407) using the income‑tax rates that are in force for the relevant financial year.  This tax may be payable directly or may be collected at source under other provisions of the Act.
Sources: [{'source': 'Income_Tax_Act_2025_as_amended_by_FA_Act_2026.pdf', 'page': 508, 'score': 0.29413020610809326, 'preview': 'B = income-tax on the specified sum calculated at the rates in force in the \nfinancial year, where “specified sum” shall have the meaning assigned \nto it in section 406 or 407;\n C = amount of income-t...'}, {'source': 'Income_Tax_Act_2025_as_amended_by_FA_Act_2026.pdf', 'page': 508, 'score': 0.29413020610809326, 'preview': 'B = income-tax on the specified sum calculated at the rates in force in the \nfinancial year, where “specified sum” shall have the meaning assigned \nto it in s

In [56]:
answer = rag_advanced("what is leave policy?", rag_retriever, llm,top_k=3,min_score=0.2,return_context=True)
print(f"Answer {answer['answer']}")
print(f"Sources: {answer['sources']}")
print(f"Confidence Score: {answer['confidence']:.2f}")


Retrieving documents for query: 'what is leave policy?' with top_k=3 and score_threshold=0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents for query: 'what is leave policy?' with min_score=0.2
Answer No relevant documents found to answer the query.
Sources: []
Confidence Score: 0.00


In [58]:
answer = rag_advanced("In what cases penalty cannot be imposed?", rag_retriever, llm,top_k=3,min_score=0.1,return_context=True)
print(f"Answer {answer['answer']}")
print(f"Sources: {answer['sources']}")
print(f"Confidence Score: {answer['confidence']:.2f}")


Retrieving documents for query: 'In what cases penalty cannot be imposed?' with top_k=3 and score_threshold=0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.25it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents for query: 'In what cases penalty cannot be imposed?' with min_score=0.1


Answer **Penalty (i.e., the fine) may not be imposed when the offender has not *wilfully* attempted to evade tax, penalty or interest.**  

Under the provision quoted, the law provides two separate punishments:

1. **“(b) in any other case … with rigorous imprisonment … and with fine.”**  
   – This applies only to the specific category of offences described in clause (b).  

2. **“(2) If a person wilfully attempts … he shall be punishable with rigorous imprisonment … and shall, in the discretion of the court, also be liable to fine.”**  
   – Here the fine is **discretionary**; the court may decide **not** to levy it.

Consequently, a fine (penalty) **cannot be imposed** in situations where:

* The conduct does **not** fall within the “any other case” described in clause (b); **or**  
* The person is **not found to have wilfully attempted** to evade payment of tax, penalty or interest, and the court, exercising its discretion, chooses not to order a fine.

In short, if the offender’s 

In [59]:
answer = rag_advanced("What are conditions for claiming deduction?", rag_retriever, llm,top_k=3,min_score=0.1,return_context=True)
print(f"Answer {answer['answer']}")
print(f"Sources: {answer['sources']}")
print(f"Confidence Score: {answer['confidence']:.2f}")


Retrieving documents for query: 'What are conditions for claiming deduction?' with top_k=3 and score_threshold=0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.83it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents for query: 'What are conditions for claiming deduction?' with min_score=0.1


Answer The deduction may be claimed only when the assessee satisfies the following condition:

* **During the tax year the assessee must be carrying on a business of prospecting for, extracting or producing petroleum and/or natural gas in India, and must have entered into an agreement with the Central Government for that business.**  

Only if this condition is met can the deduction under the provision be allowed. (The provision as quoted lists only this condition; any other sub‑conditions (e.g., (b), (c) etc.) are not shown in the excerpt.)
Sources: [{'source': 'Income_Tax_Act_2025_as_amended_by_FA_Act_2026.pdf', 'page': 654, 'score': 0.27903616428375244, 'preview': 'Quantum of deduction.\n1. (1) An assessee shall be allowed deduction of,––\n ( a) the amount or aggregate of the amount deposited by the assessee in the \naccount as specified in paragraph 2; or\n ( b) 20...'}, {'source': 'Income_Tax_Act_2025_as_amended_by_FA_Act_2026.pdf', 'page': 654, 'score': 0.27903616428375244, 'prev

### Advance RAG with streaming answers

In [65]:
# ---- Advance RAG pipeline: streaming, Citations, History, Summarization --------
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever: RAGRetriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # To store conversation history
    
    def generate_answer(self, query: str, top_k: int = 5, min_score: float = 0.2,stream: bool = False,summarize: bool = False, return_context: bool = False):
        results = self.retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
        if not results:
            return {
                "answer": "No relevant documents found to answer the query.",
                "sources": [],
                "confidence": 0,
                "context": [] if return_context else None
            }
        context = "\n\n".join([doc['content'] for doc in results])
        sources = [
                {
                    'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                    'page':doc['metadata'].get('page', 'unknown'),
                    'score': doc['similarity_score'],
                    'preview': doc['content'][:200] + ("..." if len(doc['content']) > 200 else "")
                }
            for doc in results
        ]
        confidence = max([doc['similarity_score'] for doc in results])
        prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
        if stream:
            print("Streaming answer:")
            for i in range(0, len(prompt), 80):
                print(prompt[i:i+80], end='', flush=True)
                time.sleep(0.1)  # Simulate streaming delay
                
            print()
        response = self.llm.invoke([prompt.format(context=context, question=query)],stream=stream)
        answer = response.content

        # Add citations to answer
        citations = [f"{i+1} {src['source']}  {src['page']}" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations" + "\n".join(citations) if citations else answer

        summary= None
        if summarize:
            summary_prompt = f"Summarize the following answer in 2-3 sentences:\n\n{answer}"
            summary_response = self.llm.invoke([summary_prompt])
            summary = summary_response.content.strip()
        
        self.history.append({
            "query": query,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        return{
            "answer": answer_with_citations,
            "sources": sources,
            "confidence": confidence,
            "summary": summary,
            "context": context if return_context else None,
            "history": self.history
        }
    

advanced_rag = AdvancedRAGPipeline(rag_retriever, llm)
        




In [66]:
answer = advanced_rag.generate_answer("What is income tax?", top_k=3, min_score=0.2, stream=True, summarize=True, return_context=True)
print(f"Answer: {answer['answer']}")
print(f"Summary: {answer['summary']}")

Retrieving documents for query: 'What is income tax?' with top_k=3 and score_threshold=0.2
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.55it/s]

Generated embeddings with shape: (1, 384)
Streaming answer:
Use the following context to answer the question concisely.
Context:
B = income-tax on the specified sum calculated at the rates in force in the 
financial year

, where “specified sum” shall have the meaning assigned 
to it in section 406 or 407;
 C = amount of income-tax which would be deductible or collectible at source 
during the said financial year under any provision of this Act from any 
income subject to the following:—
 ( a) such income is computed before allowing any deduction admissible 
under this Act and has been taken into account in computing the 
specified sum; and
 96. Inserted by the Finance Act, 2026, w.e.f. 1-4-2026.

B = income-tax on the specified sum calculated at the rates in force in the 
financial year, where “specified sum” shall have the meaning assigned 
to it in section 406 or 407;
 C = amount of income-tax which would be deductible or collectible at source 
during the said financial year under any provision of this Act from any 
income subject to the following:—
 ( a) such income is computed before allowing any deduction admissible 
under this Act and has been taken into account in computing the 
specified sum; a